# Đóng trọng số đã gộp thành MỘT file zip (để tải một lần, khỏi tải lẻ từng file)
Gom `checkpoints/q4b_merged/*`, `checkpoints/phowl_ft/merged/*` và `SHA256SUMS_merged.txt` của notebook **merge-lora** vào **một** file `lifemate_weights.zip` (khoảng 11 GB, **không nén** vì safetensors không nén được), kèm `lifemate_weights.zip.sha256`. Không cần GPU, không cần Internet. Khoảng **10 đến 20 phút**.

**Trước khi chạy cell đầu tiên:** đặt notebook **Private**, Internet **Off**, Accelerator **None**.
**Input cần gắn (CHỈ MỘT thứ):** **output của notebook `merge-lora`** (đã Save Version; có `checkpoints/q4b_merged`, `checkpoints/phowl_ft/merged`, `SHA256SUMS_merged.txt`). Không gắn gì khác.

**Thứ tự cell:** 1 (tìm và kiểm đủ file, sha256 từng file so với `SHA256SUMS_merged.txt`), 2 (đóng zip), 3 (mở lại zip, đọc từng file trong zip và so sha256 lần nữa, ghi sha256 của zip). Xong thì **Save Version**, rồi ở tab Output tải **hai file**: `lifemate_weights.zip` và `lifemate_weights.zip.sha256`.

**Sau khi tải về máy:** giải nén zip vào một thư mục trống, rồi trong thư mục đó chạy `sha256sum -c SHA256SUMS_merged.txt` (phải báo OK cho tất cả). Hai file chính: `q4b_merged/model.safetensors` bắt đầu bằng `652e96ae…`, `phowl_ft/merged/model.safetensors` bắt đầu bằng `cffba381…`. Zip có sẵn `HUONG_DAN.txt` ghi lại các bước này.
**Riêng tư:** đây là trọng số của đội; notebook và output để **Private**, chuyển cho chủ dự án chỉ qua Google Drive riêng (không công khai).

In [ ]:
# 1: tìm output merge-lora, kiểm đủ file và đối chiếu sha256 từng file với SHA256SUMS_merged.txt. Thiếu hoặc sai thì DỪNG.
import hashlib
import os
import sys
from pathlib import Path

ROOT = Path(os.environ.get("KROOT", "/kaggle"))
IN = ROOT / "input"
WORK = ROOT / "working"


def find_under(base, rel, maxdepth=7):
    """Tìm thư mục chứa đường dẫn tương đối `rel` (giới hạn độ sâu để khỏi quét chậm)."""
    base = str(base)
    for dirpath, dirnames, _ in os.walk(base):
        depth = dirpath[len(base):].count(os.sep)
        if depth >= maxdepth:
            dirnames[:] = []
            continue
        if (Path(dirpath) / rel).is_file():
            return Path(dirpath)
    return None


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while b := f.read(1 << 24):
            h.update(b)
    return h.hexdigest()


base = find_under(IN, "checkpoints/q4b_merged/model.safetensors")
if base is None:
    sys.exit("KHÔNG thấy checkpoints/q4b_merged/model.safetensors: gắn OUTPUT của notebook merge-lora (đã Save Version) làm input.")
CK = base / "checkpoints"
SUMS = base / "SHA256SUMS_merged.txt"
print("output merge-lora:", base)
if not SUMS.is_file():
    sys.exit(f"THIẾU {SUMS}: output merge-lora phải có SHA256SUMS_merged.txt (cell M-3).")
if not (CK / "phowl_ft" / "merged" / "model.safetensors").is_file():
    sys.exit("THIẾU checkpoints/phowl_ft/merged/model.safetensors trong output merge-lora.")

# SHA256SUMS_merged.txt ghi đường dẫn tương đối so với checkpoints/ (q4b_merged/..., phowl_ft/merged/...)
entries = []
for line in SUMS.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    h, name = line.split(None, 1)
    entries.append((h.lower(), name.strip().lstrip("*")))
bad = [n for _, n in entries if not (n.startswith("q4b_merged/") or n.startswith("phowl_ft/merged/"))]
if bad:
    sys.exit(f"SHA256SUMS_merged.txt có đường dẫn lạ (không thuộc q4b_merged/ hay phowl_ft/merged/): {bad[:3]}")
for d in ("q4b_merged", "phowl_ft/merged"):
    n = sum(1 for _, p in entries if p.startswith(d + "/"))
    print(f"{d}: {n} file trong SHA256SUMS_merged.txt")

problems = 0
total = 0
for h, name in entries:
    p = CK / name
    if not p.is_file():
        print("THIẾU ", name)
        problems += 1
        continue
    got = sha256(p)
    total += p.stat().st_size
    ok = got == h
    problems += 0 if ok else 1
    print(f"{'OK   ' if ok else 'SAI  '} {name}  ({p.stat().st_size / 1e6:.1f} MB)")
print(f"tổng {total / 1e9:.2f} GB, {len(entries)} file, {problems} lỗi")

# file thật trên đĩa mà SUMS không liệt kê: báo để khỏi sót (chúng sẽ KHÔNG vào zip)
listed = {n for _, n in entries}
extra = []
for d in ("q4b_merged", "phowl_ft/merged"):
    for p in sorted((CK / d).rglob("*")):
        if p.is_file() and p.relative_to(CK).as_posix() not in listed:
            extra.append(p.relative_to(CK).as_posix())
if extra:
    print("CẢNH BÁO: file trên đĩa không nằm trong SHA256SUMS_merged.txt (sẽ KHÔNG vào zip):", extra)
if problems:
    sys.exit("Có file thiếu hoặc sai sha256: KHÔNG đóng zip. Báo lại nguyên văn.")
want = {n: h for h, n in entries}
for k in ("q4b_merged/model.safetensors", "phowl_ft/merged/model.safetensors"):
    print("sha256", k, want[k][:8] + "…")
(WORK / "pack_base.txt").write_text(str(base), encoding="utf-8")

In [ ]:
# 2: đóng MỘT zip (không nén: ZIP_STORED, vì safetensors không nén được; zip64 tự bật). Chỉ gồm file có trong SHA256SUMS_merged.txt.
import os
import time
import zipfile
from pathlib import Path

ROOT = Path(os.environ.get("KROOT", "/kaggle"))
WORK = ROOT / "working"
base = Path((WORK / "pack_base.txt").read_text(encoding="utf-8"))
CK = base / "checkpoints"
SUMS = base / "SHA256SUMS_merged.txt"
names = []
for line in SUMS.read_text(encoding="utf-8").splitlines():
    if line.strip():
        names.append(line.split(None, 1)[1].strip().lstrip("*"))

HELP = """Giải nén lifemate_weights.zip vào MỘT thư mục trống, rồi trong thư mục đó chạy:
    sha256sum -c SHA256SUMS_merged.txt
Phải báo OK cho tất cả. Hai file chính:
    q4b_merged/model.safetensors         bắt đầu bằng 652e96ae
    phowl_ft/merged/model.safetensors    bắt đầu bằng cffba381
Kiểm cả file zip: so sha256 của lifemate_weights.zip với dòng trong lifemate_weights.zip.sha256.
Đây là trọng số của đội: chỉ chuyển qua Google Drive riêng, không công khai.
"""
out = WORK / "lifemate_weights.zip"
if out.exists():
    out.unlink()
t0 = time.time()
with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as z:
    for n in names:
        z.write(CK / n, n)
        print(f"+ {n}  ({(CK / n).stat().st_size / 1e6:.1f} MB)", flush=True)
    z.write(SUMS, "SHA256SUMS_merged.txt")
    z.writestr("HUONG_DAN.txt", HELP)
print(f"\n{out}: {out.stat().st_size / 1e9:.2f} GB, {len(names) + 2} mục, {time.time() - t0:.0f} s")

In [ ]:
# 3: mở lại zip, đọc từng file TRONG zip và so sha256 với SHA256SUMS_merged.txt (không giải nén ra đĩa), rồi ghi sha256 của chính file zip.
import hashlib
import os
import sys
import zipfile
from pathlib import Path

ROOT = Path(os.environ.get("KROOT", "/kaggle"))
WORK = ROOT / "working"
out = WORK / "lifemate_weights.zip"
bad = 0
with zipfile.ZipFile(out) as z:
    sums = z.read("SHA256SUMS_merged.txt").decode("utf-8")
    want = {}
    for line in sums.splitlines():
        if line.strip():
            h, n = line.split(None, 1)
            want[n.strip().lstrip("*")] = h.lower()
    names = set(z.namelist())
    for n, h in want.items():
        if n not in names:
            print("THIẾU trong zip:", n)
            bad += 1
            continue
        d = hashlib.sha256()
        with z.open(n) as f:
            while b := f.read(1 << 24):
                d.update(b)
        ok = d.hexdigest() == h
        bad += 0 if ok else 1
        print(f"{'OK   ' if ok else 'SAI  '} {n}")
    print("trong zip còn:", sorted(names - set(want)))
if bad:
    sys.exit(f"{bad} file trong zip thiếu hoặc sai sha256: KHÔNG dùng zip này. Báo lại nguyên văn.")
h = hashlib.sha256()
with open(out, "rb") as f:
    while b := f.read(1 << 24):
        h.update(b)
(WORK / "lifemate_weights.zip.sha256").write_text(f"{h.hexdigest()} *lifemate_weights.zip\n", encoding="utf-8")
(WORK / "pack_base.txt").unlink(missing_ok=True)
print(f"\nzip: {out.stat().st_size / 1e9:.2f} GB, sha256 {h.hexdigest()}")
print("XONG. Save Version, rồi tải hai file ở tab Output: lifemate_weights.zip và lifemate_weights.zip.sha256")